# Fib (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_cpp.ipynb)

Chapter 14 — Recursion. Runnable notebook of `Fib`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are global variables, and the trace is switched on with `trace = true`.

## Fibonacci numbers: Fib

**Input:** a natural number $n$. **Output:** $F(n)$, with $F(0) = 0$, $F(1) = 1$ and $F(n) = F(n-1) + F(n-2)$. Naive recursive algorithm: $C(n) = 2F(n+1) - 1$ calls and $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ steps.

Pseudo-code: section 14.5 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [1]:
%%writefile fib.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = number of executions of line r (numbering of the lecture notes)
long long calls = 0;    // number of calls = nodes of the tree of recursive calls
int maxDepth = 0;       // depth of the recursion = maximum height of the stack
long long mults = 0;    // multiplications (FastPower)
long long moves = 0;    // moves (Hanoi)
int ev = 0;             // counter of the events of the trace
int stk[1000];          // the call stack: the arguments of the active calls (at most 1000)
int sp = 0;             // number of active calls
long long seen[64];     // seen[x] = number of calls with argument x
bool trace = false;     // true: print the trace of the execution

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Line | Executions\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Total steps S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // like sprintf, but returns a string
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // a call starts: its frame goes on top of the stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // the call returns: its frame leaves the stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // numbered line of the trace, with the stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // line of the trace: the call, indented one level for every active call
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

// Fib(n)
long long Fib(int n)
{
    Push(n); cnt[2]++;
    if (n <= 1)
    {
        cnt[3]++;
        if (trace) Say(Str("Fib(%d)", n), Str("base case, returns %d", n));
        Pop();
        return n;
    }
    if (trace) Say(Str("Fib(%d)", n), Str("calls Fib(%d)", n - 1));
    long long x = Fib(n - 1);                      // the two calls are separated only to print the trace
    if (trace) Say(Str("Fib(%d)", n), Str("calls Fib(%d)", n - 2));
    long long y = Fib(n - 2);
    cnt[5]++;
    if (trace) Say(Str("Fib(%d)", n), Str("returns %lld + %lld = %lld", x, y, x + y));
    Pop();
    return x + y;
}

long long FibValue(int k)                          // F(k) with a loop, to check
{
    long long a = 0, b = 1;
    for (int i = 0; i < k; i++)
    {
        long long t = a + b;
        a = b;
        b = t;
    }
    return a;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Example of the lecture notes: the execution of Fib(4), in depth-first order
    int n = 4;
    Reset();
    trace = true;
    printf("%-28s| %s\n", "call", "action");
    long long r = Fib(n);
    trace = false;
    printf("F(%d) = %lld, calls C(%d) = %lld = 2F(%d) - 1, depth = %d\n", n, r, n, calls, n + 1, maxDepth);

    // Fib(5): the same subproblems are solved many times
    printf("\n");
    n = 5;
    Reset();
    r = Fib(n);
    printf("Fib(%d) = %lld: %lld calls, depth %d, S = %lld steps\n", n, r, calls, maxDepth, Total());
    printf("  k | calls of Fib(k)\n");
    for (int k = n; k >= 0; k--) printf("%3d | %lld\n", k, seen[k]);
    PrintSteps({2, 3, 5});

    // Number of calls and steps: C(n) = 2F(n+1) - 1 and S(n) = 4F(n+1) - 2
    printf("\n");
    printf("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | depth\n");
    int values[] = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30};
    for (int v : values)
    {
        Reset();
        r = Fib(v);
        printf("%3d | %6lld | %8lld | %11lld | %8lld | %11lld | %d\n", v, r, calls, 2 * FibValue(v + 1) - 1, Total(), 4 * FibValue(v + 1) - 2, maxDepth);
    }
    return 0;
}
#endif

Writing fib.cpp


In [2]:
!g++ -std=c++17 -O2 -o fib fib.cpp && ./fib

call                        | action
Fib(4)                      | calls Fib(3)
  Fib(3)                    | calls Fib(2)
    Fib(2)                  | calls Fib(1)
      Fib(1)                | base case, returns 1
    Fib(2)                  | calls Fib(0)
      Fib(0)                | base case, returns 0
    Fib(2)                  | returns 1 + 0 = 1
  Fib(3)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(3)                    | returns 1 + 1 = 2
Fib(4)                      | calls Fib(2)
  Fib(2)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(2)                    | calls Fib(0)
    Fib(0)                  | base case, returns 0
  Fib(2)                    | returns 1 + 0 = 1
Fib(4)                      | returns 2 + 1 = 3
F(4) = 3, calls C(4) = 9 = 2F(5) - 1, depth = 4

Fib(5) = 5: 15 calls, depth 5, S = 30 steps
  k | calls of Fib(k)
  5 | 1
  4 | 1
  3 | 2
  2 | 3
  1 | 5
  0 | 3
Line | Ex

**Try it yourself.** Change $n$ and see how fast the calls grow.

In [3]:
%%writefile prova_fib.cpp
#define SOLO_LIBRERIA   // do not run the main of fib.cpp
#include "fib.cpp"

int main()
{
    int n = 6;        // <-- change n (with n > 45 it takes a while)
    Reset();
    long long r = Fib(n);
    printf("F(%d) = %lld, calls = %lld, S = %lld\n", n, r, calls, Total());
    return 0;
}

Writing prova_fib.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_fib prova_fib.cpp && ./prova_fib

F(6) = 8, calls = 25, S = 50


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).